In [1]:

import os, time, threading
from concurrent.futures import ThreadPoolExecutor
ROOTS = ['/kaggle/input/models/dfranzen/intel-qwen3.8-flash-next-w4a16-autoround/transformers/default/1',
         '/kaggle/input/intel-qwen3.8-flash-next-w4a16-autoround/transformers/default/1']
root = next(r for r in ROOTS if os.path.isdir(r))
files = sorted((os.path.join(root, f) for f in os.listdir(root) if f.endswith('.safetensors')),
               key=os.path.getsize, reverse=True)
print('root', root, 'shards', len(files), 'total GB', round(sum(map(os.path.getsize, files)) / 1e9, 1), flush=True)
CHUNK = 16 * 1024 * 1024
_tl = threading.local()

def read_range(job):
    path, off, n = job
    buf = getattr(_tl, 'buf', None)
    if buf is None:
        buf = _tl.buf = bytearray(CHUNK)
    with open(path, 'rb', buffering=0) as f:
        f.seek(off)
        return f.readinto(memoryview(buf)[:n])

def probe(paths, threads, label):
    jobs = [(p, o, min(CHUNK, os.path.getsize(p) - o)) for p in paths for o in range(0, os.path.getsize(p), CHUNK)]
    t = time.time()
    with ThreadPoolExecutor(threads) as ex:
        got = sum(ex.map(read_range, jobs))
    dt = time.time() - t
    print(f'{label}: threads={threads} {got/1e9:.2f} GB in {dt:.1f}s = {got/1e6/dt:.0f} MB/s', flush=True)

budget_end = time.time() + 1500
plan = [(1, 1), (4, 1), (16, 2), (64, 2), (128, 2)]
i = 0
for threads, nfiles in plan:
    if time.time() > budget_end or i + nfiles > len(files):
        break
    probe(files[i:i + nfiles], threads, f'shards {i}-{i + nfiles - 1}')
    i += nfiles


root /kaggle/input/models/dfranzen/intel-qwen3.8-flash-next-w4a16-autoround/transformers/default/1 shards 38 total GB 181.2
shards 0-0: threads=1 5.37 GB in 26.6s = 202 MB/s
shards 1-1: threads=4 5.37 GB in 11.0s = 487 MB/s
shards 2-3: threads=16 10.74 GB in 15.8s = 680 MB/s
shards 4-5: threads=64 10.74 GB in 15.9s = 678 MB/s
shards 6-7: threads=128 10.74 GB in 15.0s = 715 MB/s
